# Multiple Integration: Theory, Algorithms, and SymPy Implementation

This notebook introduces the math behind symbolic multiple integration and demonstrates how the concepts are implemented in the `MultipleIntegrate` package.

## Table of contents

- [0 Setup](#sec-0-setup)
- [1 Definite Integration — Review](#sec-1-definite-integration-review)
- [2 Multiple Integrals](#sec-2-multiple-integrals)
- [3 A strategy map for exact multiple integration](#sec-3-strategy-map)
- [4 Multiple Integrals vs. Iterated Integrals](#sec-4-multiple-vs-iterated)
- [5 Gamma, Beta, and Dirichlet structures](#sec-5-gamma-beta-dirichlet)
- [6 Geometry of Jacobians and change of variables](#sec-6-jacobians-change-of-variables)
- [7 Gaussian integrals, quadratic forms, and determinants](#sec-7-gaussian-integrals)
- [8 Symmetry as a simplification principle](#sec-8-symmetry)
- [9 The Layer-Cake / Co-Area Viewpoint](#sec-9-layer-cake)
- [10 Region-aware multiple integration](#sec-10-region-aware)
- [11 Moments, expectations, and parameter differentiation](#sec-11-moments)
- [12 Scaling arguments](#sec-12-scaling)
- [13 Dimensional recursion](#sec-13-dimensional-recursion)
- [14 Direct SymPy vs `MultipleIntegrate`](#sec-14-direct-sympy-vs-multipleintegrate)
- [15 Examples with `multiple_integrate`](#sec-15-examples)
- [16 Supported Families — One Example Each](#sec-16-supported-families)
- [17 Complicated-looking exact multiple integrals](#sec-17-complicated-examples)
- [18 Convergence, applications, and limitations](#sec-18-convergence-applications-limitations)
- [19 References](#sec-19-references)


## <a id="sec-0-setup"></a>0  Setup

> **Range convention.** `multiple_integrate(...)` and `region_from_ranges(...)` follow the same convention as `sympy.integrate(...)`: range tuples are written in **inner-first iterated order**. The first tuple is the innermost integral and the last tuple is the outermost integral. Note that this range order differs from other symbolic integration systems such as *Mathematica*.

In [1]:
import sympy as sp
from sympy import (
    symbols, integrate, oo, pi, exp, sin, cos, sqrt, log,
    Rational, simplify, Abs, Heaviside, Piecewise, sign, diff, gamma, det, Matrix, E
)
sp.init_printing(use_unicode=True)

x, y, z, t, r, theta, phi = symbols('x y z t r theta phi', real=True)
a, b = symbols('a b', positive=True)


## <a id="sec-1-definite-integration-review"></a>1  Definite Integration — Review

### 1.1  The Riemann integral

For a bounded function $f:[a,b]\to\mathbb{R}$ the **definite integral** is the limit of Riemann sums:

$$\int_a^b f(x)\,dx \;=\; \lim_{n\to\infty}\sum_{k=1}^n f(x_k^*)\,\Delta x_k$$

### 1.2  The Fundamental Theorem of Calculus

If $F'=f$ on $[a,b]$:

$$\int_a^b f(x)\,dx = F(b)-F(a)$$

### 1.3  Improper integrals

When the domain or integrand is unbounded the integral is a limit:

$$\int_a^\infty f(x)\,dx = \lim_{R\to\infty}\int_a^R f(x)\,dx$$

**Classic example** — the Gaussian integral, proved by squaring and converting to polar coordinates:

$$\int_{-\infty}^\infty e^{-x^2}\,dx = \sqrt{\pi}$$

### 1.1  What a definite integral means

For a continuous function $f$ on an interval $[a,b]$, the definite integral

$$
\int_a^b f(x)\,dx
$$

measures the signed accumulation of $f$ over the interval.  Depending on context, the same quantity may be interpreted as:

- signed area under a curve,
- accumulated mass when $f$ is a density,
- total change when $f$ is a rate,
- probability mass when $f$ is a probability density.

The fundamental theorem of calculus gives the bridge between antiderivatives and definite integrals:

$$
\int_a^b f(x)\,dx = F(b)-F(a), \qquad F'(x)=f(x).
$$

In symbolic integration, this is only part of the story: one must also decide whether the antiderivative exists in a usable closed form, whether the integral converges, and whether a change of variables or a special-function representation is more natural than a direct antiderivative.

### 1.2  Standard tools for definite integrals

A few recurring principles show up throughout this notebook.

**Linearity**
$$
\int_a^b (\alpha f(x)+\beta g(x))\,dx
=
\alpha \int_a^b f(x)\,dx + \beta \int_a^b g(x)\,dx.
$$

**Additivity over intervals**
$$
\int_a^b f(x)\,dx = \int_a^c f(x)\,dx + \int_c^b f(x)\,dx.
$$

**Symmetry on symmetric domains**
If $f$ is odd, then
$$
\int_{-a}^{a} f(x)\,dx = 0.
$$
If $f$ is even, then
$$
\int_{-a}^{a} f(x)\,dx = 2\int_0^a f(x)\,dx.
$$

**Substitution**
If $x=\phi(u)$ is smooth and monotone, then
$$
\int_a^b f(x)\,dx
=
\int_{\phi^{-1}(a)}^{\phi^{-1}(b)} f(\phi(u))\,\phi'(u)\,du.
$$

**Integration by parts**
$$
\int_a^b u(x)\,v'(x)\,dx
=
[u(x)v(x)]_a^b - \int_a^b u'(x)\,v(x)\,dx.
$$

These rules remain important in the multivariable setting, but the domain geometry becomes an additional central ingredient.

### 1.3  Proper vs. improper integrals

Not every definite integral is taken over a finite interval or with a bounded integrand.  Typical improper integrals include

$$
\int_1^\infty \frac{dx}{x^2},
\qquad
\int_0^1 \frac{dx}{\sqrt{x}},
\qquad
\int_{-\infty}^{\infty} e^{-x^2}\,dx.
$$

These are defined by limits.  For example,

$$
\int_1^\infty \frac{dx}{x^2}
=
\lim_{R\to\infty}\int_1^R \frac{dx}{x^2}.
$$

Convergence is part of the mathematical problem, not something automatic.  A symbolic integration engine therefore has to distinguish at least three situations:

1. the integral converges and has an exact closed form,
2. the integral diverges,
3. the integral may require a special interpretation (principal value, oscillatory regularization, and so on).

This notebook focuses on ordinary convergence and exact symbolic evaluation.

In [2]:
# Verify the Gaussian integral
integrate(exp(-x**2), (x, -oo, oo))


In [3]:
# A selection of closed-form definite integrals
for expr, label in [
    (x**3,           "[0,2] polynomial"),
    (sin(x),         "[0,π] trig"),
    (exp(-a*x),      "[0,∞) exponential"),
    (1/(1 + x**2),   "(-∞,∞) rational"),
    (log(x),         "[0,1] log singularity"),
]:
    result = integrate(expr, (x, 0, 2) if label.startswith("[0,2]") else
                             (x, 0, pi) if "π]" in label else
                             (x, 0, oo) if "∞)" in label else
                             (x, -oo, oo) if "(-∞" in label else
                             (x, 0, 1))
    print(f"{label:25s}: {result}")


[0,2] polynomial         : 4
[0,π] trig               : 2
[0,∞) exponential        : 1/a
(-∞,∞) rational          : pi/2
[0,1] log singularity    : -1


## <a id="sec-2-multiple-integrals"></a>2  Multiple Integrals

### 2.1  Definition

Let $\Omega\subseteq\mathbb{R}^n$ be measurable and $f:\Omega\to\mathbb{R}$ integrable. Partitioning $\Omega$ into sub-regions $\{\Omega_k\}$ with volumes $|\Omega_k|$:

$$\int_\Omega f(\mathbf{x})\,d\mathbf{x} \;=\; \lim_{\|\text{partition}\|\to 0}\sum_k f(\mathbf{x}_k^*)\,|\Omega_k|$$

### 2.2  Applications

| Application | Integral |
|---|---|
| **Area / Volume** | $\int_\Omega 1\,d\mathbf{x}$ |
| **Mass** | $\int_\Omega \rho(\mathbf{x})\,d\mathbf{x}$ |
| **Centre of mass** | $\bar{x}_i = \frac{1}{M}\int_\Omega x_i\,\rho\,d\mathbf{x}$ |
| **Moment of inertia** | $I=\int_\Omega r^2\rho\,d\mathbf{x}$ |
| **Probability** | $\Pr[X\in A]=\int_A f_X\,d\mathbf{x}$ |
| **Partition function** | $Z=\int e^{-\beta H}\,d\mathbf{x}$ |

### 2.3  Coordinate changes

For a diffeomorphism $\mathbf{x}=\Phi(\mathbf{u})$:

$$\int_\Omega f(\mathbf{x})\,d\mathbf{x} = \int_{\Phi^{-1}(\Omega)} f(\Phi(\mathbf{u}))\,|\det J_\Phi|\,d\mathbf{u}$$

**Polar** $(r,\theta)$: $|J|=r$ &emsp; **Cylindrical** $(r,\theta,z)$: $|J|=r$ &emsp; **Spherical** $(r,\theta,\phi)$: $|J|=r^2\sin\theta$

### 2.1  Multiple integrals as integrals over regions

A multiple integral is an integral over a subset $R \subset \mathbb{R}^n$:

$$
\int_R f(\mathbf{x})\,d\mathbf{x}.
$$

For $n=2$, one usually writes
$$
\iint_R f(x,y)\,dA,
$$
and for $n=3$,
$$
\iiint_R f(x,y,z)\,dV.
$$

Geometrically, $f$ may represent:

- a density,
- a temperature field,
- a probability density,
- a charge or mass distribution,
- or simply an integrand whose total accumulation over the region is sought.

The region $R$ is just as important as the integrand.  A rectangle, a triangle, a disk, and all of $\mathbb{R}^n$ lead to very different exact methods even when the integrand itself looks similar.

### 2.2  Product regions and iterated integrals

On a rectangular region
$$
R = [a_1,b_1]\times\cdots\times[a_n,b_n],
$$
one often rewrites the multiple integral as an iterated integral:

$$
\int_R f(\mathbf{x})\,d\mathbf{x}
=
\int_{a_1}^{b_1}\cdots\int_{a_n}^{b_n} f(x_1,\dots,x_n)\,dx_n\cdots dx_1.
$$

When absolute integrability holds, Fubini's theorem says that the order of integration may be changed without affecting the result.  Tonelli's theorem gives a related statement for nonnegative integrands.

This matters computationally because different orders of integration can have very different symbolic complexity.  One order may expose a Gaussian, beta, or rational integral immediately, while another may produce difficult intermediate expressions.

### 2.3  Non-rectangular regions and dependent bounds

Not all multiple integrals live on product regions.  A triangle such as

$$
R=\{(x,y): 0\le x\le 1,\; 0\le y\le 1-x\}
$$

is still perfectly standard, and may be written as

$$
\int_0^1\int_0^{1-x} f(x,y)\,dy\,dx.
$$

The inner bound depends on the outer variable, so the geometry of the region is encoded directly into the iterated integral.  In general, changing the order of integration then requires rewriting the region, and that can be easy, moderate, or difficult depending on the boundary equations.

This notebook uses such examples to show the distinction between:
- a multiple integral as a geometric object,
- an iterated integral as one chosen description of that object.

### 2.4  Change of variables and Jacobians

For many exact multiple integrals, the key step is a coordinate transformation.  If $\mathbf{x}=\Phi(\mathbf{u})$ is a smooth bijection with Jacobian matrix $D\Phi$, then

$$
\int_R f(\mathbf{x})\,d\mathbf{x}
=
\int_{\Phi^{-1}(R)} f(\Phi(\mathbf{u}))\,\left|\det D\Phi(\mathbf{u})\right|\,d\mathbf{u}.
$$

The factor
$$
\left|\det D\Phi(\mathbf{u})\right|
$$
is the Jacobian determinant.

Examples:
- polar coordinates contribute a factor of $r$,
- cylindrical coordinates contribute a factor of $r$,
- spherical coordinates contribute a factor of $r^2\sin\theta$.

In symbolic computation, a good change of variables can turn a difficult region into a simple one, or can transform a coupled integrand into a separable product.

### 2.5  Families that often admit exact evaluation

A large fraction of exact multiple-integration problems fall into a handful of recognizable families:

- **box moments** on rectangular domains,
- **simplex moments** on triangular or simplex-shaped regions,
- **Gaussian moments** on $\mathbb{R}^n$,
- **radial integrals** over disks and balls,
- **separable product integrals**,
- **beta/gamma-type integrals**,
- **rational integrals** with contour or residue methods,
- **layer-cake or co-area reductions** for $f(g(\mathbf{x}))$.

The `multiple_integrate` function tries to recognize several of these families before falling back to more generic symbolic iteration.

In [4]:
# Volume of a ball of radius R in spherical coordinates
R = symbols('R', positive=True)
vol_ball = integrate(r**2 * sin(theta), (r, 0, R), (theta, 0, pi), (phi, 0, 2*pi))
print("Volume of ball:", vol_ball)   # 4πR³/3


Volume of ball: 4*pi*R**3/3


In [5]:
# Mass of unit disk with density ρ(r) = 1 - r²
mass = integrate((1 - r**2) * r, (r, 0, 1), (theta, 0, 2*pi))
print("Mass of unit disk:", mass)    # π/2


Mass of unit disk: pi/2


## <a id="sec-3-strategy-map"></a>3  A strategy map for exact multiple integration

The most useful mental model for this notebook is that **complicated-looking multiple integrals often become easy once one recognizes the right structural family**. In practice, a symbolic engine rarely succeeds by brute-force antiderivatives alone; it succeeds by identifying geometry, symmetry, and canonical transforms.

### 3.1  Canonical families and the methods they suggest

| Family | Typical form | Main idea | Typical output |
|---|---|---|---|
| Product regions | $\int_{[a,b]^n}\prod_i f_i(x_i)\,d\mathbf{x}$ | Separate variables and factor | Product of 1-D integrals |
| Simplex / Dirichlet | $\int_{\sum x_i\le 1}\prod x_i^{a_i-1}(1-\sum x_i)^{a_{n+1}-1}\,d\mathbf{x}$ | Recognize Beta/Dirichlet structure | Gamma ratio |
| Disks / balls / spheres | $\int_{B_R} F(\|x\|)\,dx$ | Polar, cylindrical, or spherical change of variables | Jacobian factor $\;r^{n-1}$ plus Beta/Gamma terms |
| Gaussian quadratic forms | $\int_{\mathbb{R}^n} p(x)e^{-x^TAx}\,dx$ | Complete the square, diagonalize, use moments | Determinants and Gaussian moments |
| Oscillatory–exponential cases | $\int_0^\infty e^{-ax}\cos(bx)\,dx$ and relatives | Laplace/Fourier reasoning | Rational or special-function form |
| Level-set / positive-part cases | $\int_\Omega f(g(x))\,dx$ | Layer-cake or co-area viewpoint | 1-D integral over level values |

This is the main reason `multiple_integrate` can outperform naive repeated integration on structured examples: it can try to **recognize the family first, and only then compute**.

### 3.2  Recognition before computation

For a new integral, useful questions include:

1. Is the **region** a product region, simplex, disk, ball, or graph region?
2. Is the integrand **separable**, **radial**, **Gaussian**, or **odd under a symmetry**?
3. Can a **change of variables** turn the region or integrand into a standard form?
4. Is the integral better viewed through **superlevel sets** or **pushforward densities** than through direct antiderivatives?

These questions are not merely pedagogical. They strongly predict which symbolic strategy is likely to succeed.

### 3.3  Comparison with direct iterated integration

Direct repeated integration is often enough for small examples, but it has clear limits. A structure-driven method can:

- factor product integrals immediately;
- replace simplex or ball integrals with Beta/Gamma evaluations;
- use symmetry to prove an integral is zero without computing antiderivatives;
- reduce some multidimensional problems to one-dimensional level-set integrals.

That is the organizing philosophy of the notebook sections that follow.

## <a id="sec-4-multiple-vs-iterated"></a>4  Multiple Integrals vs. Iterated Integrals

### 4.1  The distinction

A **multiple integral** $\int_\Omega f\,d\mathbf{x}$ is defined directly as a Riemann-sum limit.

An **iterated integral** reduces it to a sequence of 1-D integrals:

$$\int_a^b\!\left[\int_{g(x)}^{h(x)} f(x,y)\,dy\right]dx$$

### 4.2  Fubini's theorem

> **Fubini (1907).** If $f\in L^1([a,b]\times[c,d])$:
> $$\iint f\,dA = \int_a^b\!\int_c^d f\,dy\,dx = \int_c^d\!\int_a^b f\,dx\,dy$$

Fubini is the **licence** to compute multiple integrals as iterated ones.

### 4.3  When Fubini fails

Fubini requires $\int|f|<\infty$. The classic counterexample:

$$f(x,y)=\frac{x^2-y^2}{(x^2+y^2)^2}$$

gives $\int_0^1\!\int_0^1 f\,dy\,dx=\pi/4$ but $\int_0^1\!\int_0^1 f\,dx\,dy=-\pi/4$.

### 4.4  Order of integration on non-rectangular domains

On a triangle $\{0\le x\le 1,\;0\le y\le 1-x\}$:

$$\iint f\,dA = \int_0^1\!\int_0^{1-x}f\,dy\,dx = \int_0^1\!\int_0^{1-y}f\,dx\,dy$$

### 4.5  Tonelli versus Fubini

Two theorems are especially important in symbolic multiple integration.

- **Tonelli's theorem** applies to **nonnegative** measurable integrands. It allows one
  to exchange the order of integration even when the total integral may be infinite.
- **Fubini's theorem** applies when the integrand is **absolutely integrable**. In that
  case one may compute the multiple integral by any iterated order and obtain the same
  finite answer.

For a symbolic engine, this distinction matters because order switching, factorization,
and decomposition are only mathematically safe under appropriate convergence hypotheses.
A system that recognizes positivity, symmetry, or absolute convergence can justify
aggressive simplifications that would be unsafe for merely conditionally convergent
integrals.

In [6]:
# Fubini check — both orders must agree
f_xy = x**2 * y + x * y**3
order1 = integrate(integrate(f_xy, (y, 0, 1)), (x, 0, 1))
order2 = integrate(integrate(f_xy, (x, 0, 1)), (y, 0, 1))
print("dy dx:", order1, "  dx dy:", order2, "  equal:", simplify(order1 - order2) == 0)


dy dx: 7/24   dx dy: 7/24   equal: True


In [7]:
# Switching integration order on a triangle
order_A = integrate(integrate(x + y, (y, 0, x)),   (x, 0, 1))
order_B = integrate(integrate(x + y, (x, y, 1)),   (y, 0, 1))
print("Order A:", order_A, "  Order B:", order_B, "  equal:", simplify(order_A - order_B) == 0)


Order A: 1/2   Order B: 1/2   equal: True


## <a id="sec-5-gamma-beta-dirichlet"></a>5  Gamma, Beta, and Dirichlet structures

A striking amount of exact multiple integration reduces to a small set of special functions.

### 5.1  The Gamma and Beta functions

The **Gamma function** is
$$
\Gamma(a)=\int_0^\infty x^{a-1}e^{-x}\,dx,\qquad a>0,
$$
and the **Beta function** is
$$
B(a,b)=\int_0^1 x^{a-1}(1-x)^{b-1}\,dx
=\frac{\Gamma(a)\Gamma(b)}{\Gamma(a+b)}.
$$

These appear constantly because they are the canonical exact forms for:
- exponential tails on $[0,\infty)$;
- power weights on finite intervals;
- radial Jacobians after polar or spherical substitutions.

### 5.2  Why they appear in multiple integrals

If the region is a box and the integrand factors, then a multidimensional integral becomes a product of one-dimensional Gamma/Beta integrals. For example,
$$
\int_{(0,\infty)^n}\prod_{i=1}^n x_i^{a_i-1}e^{-b_i x_i}\,d\mathbf{x}
=
\prod_{i=1}^n \frac{\Gamma(a_i)}{b_i^{a_i}}.
$$

Likewise, power-law integrands on $[0,1]^n$ naturally produce Beta factors.

### 5.3  Dirichlet integrals on simplices

For simplex regions, the correct higher-dimensional analogue is the **Dirichlet integral**:
$$
\int_{\substack{x_i\ge 0\\x_1+\cdots+x_n\le 1}}
x_1^{a_1-1}\cdots x_n^{a_n-1}(1-x_1-\cdots-x_n)^{a_{n+1}-1}\,d\mathbf{x}
=
\frac{\prod_{j=1}^{n+1}\Gamma(a_j)}{\Gamma(a_1+\cdots+a_{n+1})}.
$$

This formula explains why many apparently complicated simplex integrals in the notebook collapse to a clean Gamma ratio.

## <a id="sec-6-jacobians-change-of-variables"></a>6  Geometry of Jacobians and change of variables

A good change of variables often does two jobs at once: it makes the **region simpler** and the **integrand more canonical**.

### 6.1  Jacobians encode local volume scaling

If $x=\Phi(u)$ is a smooth invertible substitution, then
$$
\int_{\Phi(U)} f(x)\,dx = \int_U f(\Phi(u))\,\left|\det D\Phi(u)\right|\,du.
$$

The determinant measures how small volume elements are stretched or compressed by the map.

### 6.2  Why polar and spherical coordinates produce powers of $r$

In two dimensions,
$$
x=r\cos\theta,\qquad y=r\sin\theta,
$$
and the Jacobian contributes a factor of $r$. In three dimensions, spherical coordinates contribute $r^2\sin\theta$. More generally, radial integration in $\mathbb{R}^n$ produces the factor $r^{n-1}$.

That single Jacobian factor is what turns many disk and ball integrals into Beta- or Gamma-type integrals.

### 6.3  Linear substitutions and determinants

For linear maps $x=Au$, the Jacobian is constant:
$$
dx = |\det A|\,du.
$$
This is why ellipses, ellipsoids, and rotated quadratic forms are often tractable symbolically: one can map them to disks, balls, or diagonal quadratic forms and pull out a determinant factor.

A natural next extension for `MultipleIntegrate` is to reuse the chart and Jacobian machinery from the earlier `symcoord` work. That would allow the solver to treat several important families by an explicit coordinate change before fallback: polar coordinates on disks and sectors, spherical or cylindrical coordinates on balls and shells, affine-linear maps for ellipses and tilted simplices, and linear diagonalization for quadratic-form Gaussian integrals. In practice, the most useful near-term targets are integrands of the form $x^a y^b f(x^2+y^2)$ on disks and analogous spherical moment integrals in three dimensions. Those are not purely radial, but they become separable after a standard change of variables.


## <a id="sec-7-gaussian-integrals"></a>7  Gaussian integrals, quadratic forms, and determinants

Gaussian integrals are one of the cleanest bridges between symbolic integration, linear algebra, and probability.

### 7.1  The basic determinant formula

If $A$ is symmetric positive definite, then
$$
\int_{\mathbb{R}^n} e^{-x^T A x}\,dx
=
\frac{\pi^{n/2}}{\sqrt{\det A}}.
$$

This formula explains why exact Gaussian integrals are controlled by **determinants** rather than by complicated repeated antiderivatives.

### 7.2  Moments and polynomial factors

When a polynomial factor is present,
$$
\int_{\mathbb{R}^n} p(x)e^{-x^TAx}\,dx,
$$
the result can often be obtained by symmetry, parameter differentiation, or reduction to one-dimensional Gaussian moments.

Odd monomials often vanish on symmetric regions, while even monomials reduce to products of standard Gaussian moments.

### 7.3  Why quadratic-form recognition matters

A symbolic engine that recognizes quadratic forms can:
- diagonalize or complete the square;
- exploit symmetry immediately;
- reduce the problem to determinant and moment formulas.

That is why Gaussian examples often look harder than they really are.

## <a id="sec-8-symmetry"></a>8  Symmetry as a simplification principle

One of the fastest ways to compute a multiple integral is to prove that most of it cancels.

### 8.1  Oddness on symmetric regions

If a region is symmetric under $x_i\mapsto -x_i$ and the integrand is odd in $x_i$, then the integral is zero. For example,
$$
\int_{-a}^a \int_{-b}^b x\,g(x^2,y^2)\,dy\,dx = 0.
$$

This principle extends directly to disks, balls, and full-space Gaussian integrals.

### 8.2  Rotational symmetry and equal moments

On a rotationally symmetric region,
$$
\int_\Omega x_1^2\,dx
=
\int_\Omega x_2^2\,dx
=\cdots=
\int_\Omega x_n^2\,dx,
$$
and
$$
\int_\Omega \|x\|^2\,dx
=
n\int_\Omega x_1^2\,dx.
$$

These identities reduce multidimensional moment computations to a single representative coordinate.

### 8.3  Why symbolic systems benefit from symmetry detection

Symmetry is not only elegant; it is computationally powerful. It can:
- prove vanishing results immediately;
- reduce the number of distinct moments that must be computed;
- justify simpler coordinate choices and fewer cases in piecewise geometry.

## <a id="sec-9-layer-cake"></a>9  The Layer-Cake / Co-Area Viewpoint

`multiple_integrate` reduces an $n$-dimensional integral to a **one-dimensional one**
for integrands of the form $f(g(\mathbf{x}))$.  
**Crucially, $g$ can be any measurable function — polynomial, trigonometric,
exponential, or discontinuous.**

### 9.1  The master identity

Define the **cumulative measure** of $g$ over $\Omega$:

$$\mu(y) = \lambda\!\left(\{\mathbf{x}\in\Omega : g(\mathbf{x})\le y\}\right)
= \int_\Omega \Theta(y - g(\mathbf{x}))\,d\mathbf{x}$$

where $\Theta$ is the Heaviside step function. Then:

$$\boxed{
\int_\Omega f(g(\mathbf{x}))\,d\mathbf{x} = \int_{y_{\min}}^{y_{\max}} f(y)\,\mu'(y)\,dy
}$$

This follows from Fubini's theorem and requires **nothing special about $g$**.

### 9.2  The co-area formula (Federer, 1959)

$$\mu'(y) = \int_{g^{-1}(y)} \frac{1}{|\nabla g(\mathbf{x})|}\,d\mathcal{H}^{n-1}(\mathbf{x})$$

For a **single-variable monotone** $g$, the level set $g^{-1}(y)$ is one point $x(y)$, so:

$$\mu'(y) = \frac{1}{|g'(x(y))|} = \left|\frac{dx}{dy}\right|$$

For **piecewise-monotone** $g$ with branches $\{x_k\}$ at each level $y$:

$$\mu'(y) = \sum_k \frac{1}{|g'(x_k)|}$$

### 9.3  Special cases

| $g$ | Domain | $\mu'(y)$ | Strategy |
|---|---|---|---|
| $\mathbf{b}\cdot\mathbf{x}+c$ (linear) | $[0,\infty)^n$ | $\frac{(y-c)^{n-1}}{\prod b_i\,(n-1)!}$ | S1 |
| $\mathbf{x}^\top A\mathbf{x}$ (quadratic) | $\mathbb{R}^n$ | $\frac{\pi^{n/2}}{\sqrt{\det A}\,\Gamma(n/2+1)}\cdot\frac{n}{2}(y-y_{\min})^{n/2-1}$ | S2/S3 |
| any polynomial | bounded $\Omega$ | computed by SymPy from $\int\Theta(y-g)$ | S4 |
| $h_1(x_1)+\cdots+h_n(x_n)$ | any | convolution $\nu_1*\cdots*\nu_n$ | S5 |
| monotone $g(x_i)$ | $[a_i,b_i]$ | $\lvert dx/dy\rvert$ via $g^{-1}$ | S6 |
| piecewise-monotone $g(x_i)$ | $[a_i,b_i]$ | $\sum_k 1/\lvert g'(x_k)\rvert$ | S7 |
| non-polynomial multivariate | bounded | SymPy evaluates $\int\Theta(y-g)$ | S8 |

### 9.4  Step-by-step walkthrough — polynomial $g$

We demonstrate the general Heaviside approach (Strategy 4) on $\iint_{[0,1]^2}(x+y)^3\,dA$.

In [8]:
# The level sets of g = x+y on [0,1]^2 are line segments.
# We derive mu(y) analytically:
#   For y in [0,1]: {x+y <= y} is the triangle below the line, area = y^2/2
#   For y in [1,2]: area = 1 - (2-y)^2/2
ys = symbols('ys', real=True)

mu_poly = Piecewise(
    (ys**2 / 2,           (ys >= 0) & (ys <= 1)),
    (1 - (2 - ys)**2 / 2, (ys > 1)  & (ys <= 2)),
    (0, True)
)
print("μ(y) =", mu_poly)


μ(y) = Piecewise((ys**2/2, (ys >= 0) & (ys <= 1)), (1 - (2 - ys)**2/2, (ys <= 2) & (ys > 1)), (0, True))


In [9]:
# Step 2: differentiate to get the pushforward density μ'(y)
density_poly = simplify(diff(mu_poly, ys))
print("μ'(y) =", density_poly)
# Triangular density: rises linearly to 1 at y=1, falls back to 0 at y=2


μ'(y) = Piecewise((ys, (ys >= 0) & (ys <= 1)), (2 - ys, (ys <= 2) & (ys > 1)), (0, True))


In [10]:
# Step 3: ∫ f(y)·μ'(y) dy  with f(t) = t³
result_layercake = integrate(ys**3 * density_poly, (ys, 0, 2))
result_direct    = integrate(integrate((x + y)**3, (y, 0, 1)), (x, 0, 1))
print("Layer-cake:", result_layercake)   # 3/2
print("Direct:    ", result_direct)      # 3/2
print("Match:", simplify(result_layercake - result_direct) == 0)


Layer-cake: 3/2
Direct:     3/2
Match: True


### 9.5  Step-by-step walkthrough — non-polynomial $g$

The same identity works for $g = \sin(x)$ on $[0,\pi]$, a piecewise-monotone function.

We derive $\mu(y)$ analytically. For $y \in [0,1]$, the set
$\{x\in[0,\pi] : \sin(x) \le y\}$ consists of two pieces:
$[0, \arcsin y]$ and $[\pi - \arcsin y, \pi]$, with total length $2\arcsin y$.
Differentiating:

$$\mu'(y) = \frac{2}{\sqrt{1-y^2}}, \qquad y \in [0,1]$$

This is the sum of two co-area branches: $1/|\sin'(x_1)| + 1/|\sin'(x_2)|
= 1/\cos(\arcsin y) + 1/\cos(\pi-\arcsin y) = 2/\sqrt{1-y^2}$.


In [11]:
# μ(y) for g = sin(x) on [0, π] derived analytically
ys2 = symbols('ys2', real=True)

mu_sin = Piecewise(
    (0,           ys2 < 0),
    (2*sp.asin(ys2), (ys2 >= 0) & (ys2 <= 1)),
    (pi,          ys2 > 1)
)
density_sin = simplify(diff(mu_sin, ys2))
print("μ'(y) =", density_sin)   # 2/sqrt(1 - y^2) on [0,1]

# For f = identity: ∫_0^1 y · μ'(y) dy should equal ∫_0^π sin(x) dx = 2
result_sin = integrate(ys2 * density_sin, (ys2, 0, 1))
print("Layer-cake ∫sin(x):", result_sin)   # 2
print("Direct     ∫sin(x):", integrate(sin(x), (x, 0, pi)))  # 2
print("Match:", simplify(result_sin - 2) == 0)


μ'(y) = Piecewise((0, ys2 < 0), (2/sqrt(1 - ys2**2), ys2 <= 1), (0, True))
Layer-cake ∫sin(x): 2
Direct     ∫sin(x): 2
Match: True


## <a id="sec-10-region-aware"></a>10  Region-aware multiple integration

The solver now has an explicit **region layer** in front of several exact strategies. Input ranges are normalized into region objects, and those regions control symmetry checks, moment formulas, safe dependent-bound handling, coordinate-change selection, and some reversal logic.


### 10.1  Current region types

At the moment the package can recognize and use these structured region types:

- `BoxRegion`
- `IteratedRegion`
- `SimplexRegion`
- `GraphRegion`
- `DiskRegion`
- `BallRegion`

This does **not** make the package a full symbolic region engine, but it substantially improves how several multiple-integral families are handled.

In [12]:
import sys
from pathlib import Path

def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "multiple_integrate").exists():
            return candidate
    return start

ROOT = _find_project_root(Path.cwd())
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", ROOT)
print("Using src path:", SRC)
print("Package expected at:", SRC / "multiple_integrate")

from multiple_integrate import multiple_integrate
from multiple_integrate.regions import region_from_ranges


Project root: /Users/bbhatt/Downloads/MultipleIntegrate 2
Using src path: /Users/bbhatt/Downloads/MultipleIntegrate 2/src
Package expected at: /Users/bbhatt/Downloads/MultipleIntegrate 2/src/multiple_integrate


In [13]:
R_box = region_from_ranges([(x, 0, 1), (y, 0, 1)])
R_simplex = region_from_ranges([(y, 0, 1 - x), (x, 0, 1)])
R_disk = region_from_ranges([(y, -sp.sqrt(1 - x**2), sp.sqrt(1 - x**2)), (x, -1, 1)])

print(type(R_box).__name__)
print(type(R_simplex).__name__)
print(type(R_disk).__name__)

BoxRegion
SimplexRegion
DiskRegion


### 10.2  Simplex moments

Standard simplex-style dependent bounds are now recognized directly. This means the solver can apply exact simplex moment formulas rather than treating these only as generic nested integrals.

In [14]:
x, y, z = symbols('x y z', real=True)

simplex_area = multiple_integrate(1, (y, 0, 1 - x), (x, 0, 1))
simplex_xy = multiple_integrate(x * y, (y, 0, 1 - x), (x, 0, 1))
simplex_3d = multiple_integrate(1, (z, 0, 1 - x - y), (y, 0, 1 - x), (x, 0, 1))

print(simplex_area)   # 1/2
print(simplex_xy)     # 1/24
print(simplex_3d)     # 1/6

1/2
1/24
1/6


### 10.3  Graph regions and safe order reversal

For simple 2D affine graph regions, the solver can reverse order safely and, when necessary, split the reversed region into a small number of cells. This is intentionally limited to simple cases, but it already covers a useful class of dependent-bound multiple integrals.

In [15]:
graph_ex = multiple_integrate(y, (y, x, 1), (x, 0, 1))
print(graph_ex)  # 1/3

1/3


### 10.4  Disk and ball regions

The package now recognizes standard centered disks and balls written in nested-bounds form. This allows exact polynomial moments, selected polar / spherical coordinate changes, and several nontrivial weighted examples to be evaluated directly.


In [16]:
disk_area = multiple_integrate(1, (y, -sp.sqrt(1 - x**2), sp.sqrt(1 - x**2)), (x, -1, 1))
disk_x2 = multiple_integrate(x**2, (y, -sp.sqrt(1 - x**2), sp.sqrt(1 - x**2)), (x, -1, 1))

ball_vol = multiple_integrate(
    1,
    (z, -sp.sqrt(1 - x**2 - y**2), sp.sqrt(1 - x**2 - y**2)),
    (y, -sp.sqrt(1 - x**2), sp.sqrt(1 - x**2)),
    (x, -1, 1),
)

print(disk_area)  # pi
print(disk_x2)    # pi/4
print(ball_vol)   # 4*pi/3

pi
pi/4
4*pi/3


### 10.5  Radial and angular-factor integrals on disks and balls

For standard centered disks and balls, the solver no longer stops at purely radial integrands. It can also recognize some integrands that become separable after a polar or spherical change of variables, including factors such as $x^a y^b f(x^2+y^2)$ on disks and analogous moment-style families in three dimensions.


In [17]:
radial_disk = multiple_integrate(
    sp.exp(-(x**2 + y**2)),
    (x, -1, 1),
    (y, -sp.sqrt(1 - x**2), sp.sqrt(1 - x**2)),
)

radial_ball = multiple_integrate(
    sp.exp(-(x**2 + y**2 + z**2)),
    (z, -sp.sqrt(1 - x**2 - y**2), sp.sqrt(1 - x**2 - y**2)),
    (y, -sp.sqrt(1 - x**2), sp.sqrt(1 - x**2)),
    (x, -1, 1),
)

print(sp.simplify(radial_disk))
print(sp.simplify(radial_ball))

pi*erf(1)*erf(sqrt(1 - x**2))
-2*pi*exp(-1) + pi**(3/2)*erf(1)


## <a id="sec-11-moments"></a>11  Moments, expectations, and parameter differentiation

Many structured multiple integrals are really **moments** of a measure, density, or distribution.  This viewpoint unifies several of the solver's strongest families.

Typical examples include:

- **geometric moments** over regions such as disks, balls, and simplices,
- **Gaussian moments** over $\mathbb{R}^n$,
- **Beta / Dirichlet moments** on intervals and simplices,
- **expectations** of random variables in probability.

For a region $R$ with density $\rho(x)$, an integral of the form

$$
\int_R x_1^{a_1}\cdots x_n^{a_n}\,\rho(x)\,dx
$$

is a moment.  In probability language, if $\rho$ is normalized to integrate to $1$, then

$$
\mathbb{E}[X_1^{a_1}\cdots X_n^{a_n}]
=
\int x_1^{a_1}\cdots x_n^{a_n}\,\rho(x)\,dx.
$$

This is one reason Gamma, Beta, and Gaussian integrals appear so often: they are the normalizing constants and moments of standard families.

In [18]:
# Moment examples across several families
x, y, z = symbols('x y z', real=True)

moment_box = multiple_integrate(x**2 * y**3, (x, 0, 1), (y, 0, 1))
moment_simplex = multiple_integrate(x * y, (y, 0, 1 - x), (x, 0, 1))
moment_gaussian = multiple_integrate(x**2 * exp(-x**2), (x, -oo, oo))

print("Box moment ∫_[0,1]^2 x^2 y^3 dx dy =", moment_box)
print("Simplex moment ∫_Δ x y dA          =", moment_simplex)
print("Gaussian moment ∫_ℝ x^2 e^(-x^2) dx =", moment_gaussian)

Box moment ∫_[0,1]^2 x^2 y^3 dx dy = 1/12
Simplex moment ∫_Δ x y dA          = 1/24
Gaussian moment ∫_ℝ x^2 e^(-x^2) dx = sqrt(pi)/2


### 11.1  Probability interpretations

Several closed forms in this notebook can be read probabilistically.

If $X \sim \mathcal{N}(0, \tfrac12)$ has density proportional to $e^{-x^2}$, then

$$
\int_{-\infty}^{\infty} x^{2k} e^{-x^2}\,dx
$$

is an unnormalized even Gaussian moment.  Likewise, on the interval $[0,1]$,

$$
\int_0^1 x^{a-1}(1-x)^{b-1}\,dx = B(a,b)
$$

is the normalizing constant of the Beta distribution.  On a simplex, Dirichlet integrals play the same role in higher dimensions.

This perspective is useful because it turns symbolic integration questions into structural questions:

- what family of density is hiding here?
- is the integral really a moment or expectation?
- does normalization isolate a known special function?

### 11.2  Differentiation under the integral sign

A classical way to generate moments is to introduce a parameter and differentiate.  For example, define

$$
I(a) = \int_{-\infty}^{\infty} e^{-a x^2}\,dx
\qquad (a>0).
$$

Since

$$
I(a)=\sqrt{\pi}\,a^{-1/2},
$$

differentiating gives

$$
I'(a)
=
-\int_{-\infty}^{\infty} x^2 e^{-a x^2}\,dx
=
-\frac12 \sqrt{\pi}\,a^{-3/2}.
$$

Therefore

$$
\int_{-\infty}^{\infty} x^2 e^{-a x^2}\,dx
=
\frac12 \sqrt{\pi}\,a^{-3/2}.
$$

This method works much more generally: once a parameterized family is known exactly, derivatives with respect to the parameter often produce new exact multiple integrals.

In [19]:
# Differentiate under the integral sign to recover a Gaussian moment
a = symbols('a', positive=True)

I = integrate(exp(-a * x**2), (x, -oo, oo))
moment_from_param = -diff(I, a)
moment_direct = integrate(x**2 * exp(-a * x**2), (x, -oo, oo))

print("I(a) =", simplify(I))
print("-d/da I(a) =", simplify(moment_from_param))
print("Direct moment =", simplify(moment_direct))
print("Agreement:", simplify(moment_from_param - moment_direct) == 0)

I(a) = sqrt(pi)/sqrt(a)
-d/da I(a) = sqrt(pi)/(2*a**(3/2))
Direct moment = sqrt(pi)/(2*a**(3/2))
Agreement: True


## <a id="sec-12-scaling"></a>12  Scaling arguments

Scaling is one of the cleanest structural tools in multiple integration.  If a region or integrand has a built-in scale, the answer often must be a constant times a power of that scale.

For instance, in $n$ dimensions,

$$
\int_{\mathbb{R}^n} e^{-a\|x\|^2}\,dx
=
a^{-n/2}
\int_{\mathbb{R}^n} e^{-\|u\|^2}\,du,
$$

after the substitution $u=\sqrt{a}\,x$.  The power $a^{-n/2}$ is forced by the Jacobian.

Similarly, if $B_R$ is the ball of radius $R$, then

$$
\operatorname{Vol}(B_R)=R^n \operatorname{Vol}(B_1).
$$

More generally, homogeneous integrands over scaled regions often reduce to a dimensional constant times a simple power of the scaling parameter.  This is both a theoretical guide and a practical simplification heuristic.

In [20]:
# Scaling of Gaussian integrals and ball volumes
a = symbols('a', positive=True)
R = symbols('R', positive=True)
u, v = symbols('u v', real=True)

gauss_scaled = integrate(exp(-a * (x**2 + y**2)), (x, -oo, oo), (y, -oo, oo))
gauss_unit = integrate(exp(-(u**2 + v**2)), (u, -oo, oo), (v, -oo, oo))

ball_area_R = integrate(1, (x, -R, R), (y, -sqrt(R**2 - x**2), sqrt(R**2 - x**2)))
ball_area_1 = integrate(1, (y, -sqrt(1 - x**2), sqrt(1 - x**2)), (x, -1, 1))

print("Scaled Gaussian in 2D:", simplify(gauss_scaled))
print("a^(-1) * unit Gaussian:", simplify(a**(-1) * gauss_unit))
print("Disk area radius R:", simplify(ball_area_R))
print("R^2 * unit disk area:", simplify(R**2 * ball_area_1))

Scaled Gaussian in 2D: pi/a
a^(-1) * unit Gaussian: pi/a
Disk area radius R: 4*R*sqrt(R**2 - x**2)
R^2 * unit disk area: pi*R**2


## <a id="sec-13-dimensional-recursion"></a>13  Dimensional recursion

Many canonical multiple integrals satisfy natural recursions in the dimension $n$.  This is another reason Gamma functions appear so often: they package those recurrences cleanly.

A standard example is the volume of the unit ball:

$$
V_n = \frac{\pi^{n/2}}{\Gamma\!\left(\frac{n}{2}+1\right)}.
$$

From the identity $\Gamma(z+1)=z\,\Gamma(z)$, one gets the recursion

$$
V_n = \frac{2\pi}{n} V_{n-2}.
$$

Likewise, Gaussian integrals satisfy

$$
\int_{\mathbb{R}^n} e^{-\|x\|^2}\,dx = \pi^{n/2},
$$

so increasing the dimension by $2$ multiplies the answer by $\pi$.

These recurrences are useful conceptually because they show that many high-dimensional formulas are not isolated miracles; they are part of coherent dimensional families.

In [21]:
# Dimensional recursion for unit-ball volumes
n = symbols('n', integer=True, positive=True)
Vn = pi**(n/2) / gamma(n/2 + 1)
recurrence_check = simplify(Vn / (2*pi/n * (pi**((n-2)/2) / gamma((n-2)/2 + 1))))

print("V_n =", Vn)
print("Check V_n = (2π/n) V_{n-2}:", recurrence_check)
print("V_1 =", simplify(Vn.subs(n, 1)))
print("V_2 =", simplify(Vn.subs(n, 2)))
print("V_3 =", simplify(Vn.subs(n, 3)))
print("V_4 =", simplify(Vn.subs(n, 4)))

V_n = pi**(n/2)/gamma(n/2 + 1)
Check V_n = (2π/n) V_{n-2}: 1
V_1 = 2
V_2 = pi
V_3 = 4*pi/3
V_4 = pi**2/2


## <a id="sec-14-direct-sympy-vs-multipleintegrate"></a>14  Direct SymPy vs `MultipleIntegrate`

`MultipleIntegrate` is not meant to replace SymPy's general-purpose `integrate`. Instead, it tries to recognize structural families early and route them to methods that are especially natural for multiple integrals.

In practice, there are now four especially important differences:

- **recognition before brute force**: product regions, simplices, disks, balls, Gaussians, and some transform-friendly forms can be handled by a family-specific method,
- **region awareness**: the package can use information from the domain, not just the raw nested antiderivative problem,
- **direct exact formulas**: Dirichlet / simplex cases can now bypass fragile repeated definite integration,
- **coordinate changes as a first-class strategy**: selected disk and ball integrals can be reduced by polar or spherical substitutions before fallback.

The comparison cells below are written to avoid long notebook stalls. For some structured examples, `sympy.integrate(...)` can take a very long time or wander into branch-sensitive hypergeometric forms, so the notebook separates:

- cases where direct SymPy comparison is usually quick,
- cases where `MultipleIntegrate` is the more appropriate demonstration,
- and one case where SymPy can return an incorrect value while `MultipleIntegrate` returns the correct exact result.



In [22]:
# Direct SymPy vs MultipleIntegrate on representative examples
import time
x, y, z, w = symbols('x y z w', real=True)


def show_comparison(label, expr, ranges, *, run_sympy=True, expected=None):
    print(label)

    sympy_res = None
    mi_res = None

    if run_sympy:
        print("  running SymPy.integrate ...", flush=True)
        t0 = time.perf_counter()
        try:
            sympy_res = simplify(integrate(expr, *ranges))
            t1 = time.perf_counter()
            print("  SymPy integrate      =", sympy_res)
            print(f"  SymPy time           = {t1 - t0:.3f} s")
        except Exception as exc:
            t1 = time.perf_counter()
            print("  SymPy integrate      = ERROR:", exc)
            print(f"  SymPy time           = {t1 - t0:.3f} s")
    else:
        print("  SymPy integrate      = skipped here (can be very slow or fragile)")

    print("  running MultipleIntegrate ...", flush=True)
    t0 = time.perf_counter()
    try:
        mi_res = simplify(multiple_integrate(expr, *ranges))
        t1 = time.perf_counter()
        print("  MultipleIntegrate    =", mi_res)
        print(f"  MultipleIntegrate time = {t1 - t0:.3f} s")
    except Exception as exc:
        t1 = time.perf_counter()
        print("  MultipleIntegrate    = ERROR:", exc)
        print(f"  MultipleIntegrate time = {t1 - t0:.3f} s")

    if expected is not None:
        print("  expected             =", expected)
        if mi_res is not None:
            try:
                print("  MI correct           =", simplify(mi_res - expected) == 0)
            except Exception:
                print("  MI correct           = undetermined")
        if sympy_res is not None:
            try:
                print("  SymPy correct        =", simplify(sympy_res - expected) == 0)
            except Exception:
                print("  SymPy correct        = undetermined")

    if sympy_res is not None and mi_res is not None:
        try:
            print("  equal                =", simplify(sympy_res - mi_res) == 0)
        except Exception:
            print("  equal                = undetermined")

    print()


fast_comparisons = [
    (
        "Simplex moment",
        x * y,
        ((y, 0, 1 - x), (x, 0, 1)),
        sp.Rational(1, 24),
    ),
    (
        "Disk polynomial moment",
        x**2,
        ((y, -sqrt(1 - x**2), sqrt(1 - x**2)), (x, -1, 1)),
        sp.pi / 4,
    ),
    (
        "Oscillatory product region",
        exp(-x - y) * cos(x - y),
        ((y, 0, oo), (x, 0, oo)),
        sp.Rational(1, 2),
    ),
]

for label, expr, ranges, expected in fast_comparisons:
    show_comparison(label, expr, ranges, run_sympy=True, expected=expected)


structured_showcases = [
    (
        "Simplex Dirichlet with fractional exponents",
        sp.sqrt(x) * y**sp.Rational(3, 2) * sp.sqrt(1 - x - y),
        ((y, 0, 1 - x), (x, 0, 1)),
        sp.gamma(sp.Rational(3, 2)) * sp.gamma(sp.Rational(5, 2)) * sp.gamma(sp.Rational(3, 2)) / sp.gamma(sp.Rational(11, 2)),
    ),
    (
        "Disk coordinate-change example",
        x**2 * y**2 / sqrt(1 - x**2 - y**2),
        ((y, -sqrt(1 - x**2), sqrt(1 - x**2)), (x, -1, 1)),
        sp.pi / 24,
    ),
]

for label, expr, ranges, expected in structured_showcases:
    show_comparison(label, expr, ranges, run_sympy=False, expected=simplify(expected))



Simplex moment
  running SymPy.integrate ...
  SymPy integrate      = 1/24
  SymPy time           = 0.018 s
  running MultipleIntegrate ...
  MultipleIntegrate    = 1/24
  MultipleIntegrate time = 0.006 s
  expected             = 1/24
  MI correct           = True
  SymPy correct        = True
  equal                = True

Disk polynomial moment
  running SymPy.integrate ...
  SymPy integrate      = pi/4
  SymPy time           = 1.537 s
  running MultipleIntegrate ...
  MultipleIntegrate    = pi/4
  MultipleIntegrate time = 0.008 s
  expected             = pi/4
  MI correct           = True
  SymPy correct        = True
  equal                = True

Oscillatory product region
  running SymPy.integrate ...
  SymPy integrate      = 1/2
  SymPy time           = 0.182 s
  running MultipleIntegrate ...
  MultipleIntegrate    = 1/2
  MultipleIntegrate time = 0.125 s
  expected             = 1/2
  MI correct           = True
  SymPy correct        = True
  equal                = True

Simpl

## <a id="sec-15-examples"></a>15  Examples with `multiple_integrate`

The library implements all nine strategies. Place `multiple_integrate` on your
`PYTHONPATH` (or install with `pip install -e .` from the project root):

In [24]:
from multiple_integrate import multiple_integrate

Before diving into the strategy-specific examples, it is helpful to keep the overall philosophy in mind. The solver is not a single monolithic rule. Instead, it tries to identify structure such as:

- linear or quadratic forms,
- Gaussian-type weights,
- product or separable structure,
- monotone substitutions,
- simplex / Dirichlet structure,
- radial, angular-factor, or moment families after a coordinate change,
- and, as a last resort, plain iterated integration.

The examples below are grouped by the dominant exact-evaluation idea rather than by increasing difficulty. In practice, many integrals fit more than one pattern.


### 15.1  Strategy 1 — Linear polynomial over $[0,\infty)^n$

**Formula:** $\displaystyle\int_{[0,\infty)^n} f(\mathbf{b}\cdot\mathbf{x}+c)\,d\mathbf{x}
= \frac{1}{\prod b_i\,(n-1)!}\int_c^\infty (y-c)^{n-1}f(y)\,dy$

Fires when $g$ is a *purely linear* polynomial and every range is $[0,\infty)$.


In [25]:
x, y, z = symbols('x y z', real=True)

# n=1: ∫_0^∞ exp(-(2x+1)) dx = exp(-1)/2
s1a = multiple_integrate(exp(-(2*x + 1)), (x, 0, oo))
print("∫₀^∞ exp(-(2x+1)) dx =", s1a)   # exp(-1)/2

# n=2: ∫∫_[0,∞)² exp(-(3x+2y)) dx dy = 1/6
s1b = multiple_integrate(exp(-(3*x + 2*y)), (x, 0, oo), (y, 0, oo))
print("∫∫ exp(-(3x+2y))    =", s1b)   # 1/6

# n=3: ∫∫∫_[0,∞)³ exp(-(x+y+z)) dV = 1
s1c = multiple_integrate(exp(-(x+y+z)), (x, 0, oo), (y, 0, oo), (z, 0, oo))
print("∫∫∫ exp(-(x+y+z))  =", s1c)   # 1


∫₀^∞ exp(-(2x+1)) dx = exp(-1)/2
∫∫ exp(-(3x+2y))    = 1/6
∫∫∫ exp(-(x+y+z))  = 1


### 15.2  Strategy 2 — Quadratic, doubly-infinite

**Formula:** $\displaystyle\int_{\mathbb{R}^n} f(\mathbf{x}^\top A\mathbf{x}+\mathbf{b}\cdot\mathbf{x}+c)\,d\mathbf{x}
= \frac{\pi^{n/2}}{\sqrt{\det A}\,\Gamma(n/2+1)}\int_{y_{\min}}^\infty \frac{n}{2}(y-y_{\min})^{n/2-1}f(y)\,dy$

Fires when $g$ is quadratic with positive-definite $A$ and all ranges are $(-\infty,\infty)$.


In [26]:
# 2-D isotropic Gaussian
s2a = multiple_integrate(exp(-(x**2 + y**2)), (x, -oo, oo), (y, -oo, oo))
print("∫∫_ℝ² exp(-(x²+y²))         =", s2a)   # π

# Anisotropic: ∫∫ exp(-(2x²+3y²)) = π/√6
s2b = multiple_integrate(exp(-(2*x**2 + 3*y**2)), (x, -oo, oo), (y, -oo, oo))
print("∫∫_ℝ² exp(-(2x²+3y²))       =", s2b)   # π/√6

# 3-D Gaussian
s2c = multiple_integrate(exp(-(x**2+y**2+z**2)), (x,-oo,oo),(y,-oo,oo),(z,-oo,oo))
print("∫∫∫_ℝ³ exp(-(x²+y²+z²))     =", s2c)   # π^(3/2)

# With shift: value is still π
s2d = multiple_integrate(exp(-((x-1)**2 + (y+2)**2)), (x,-oo,oo),(y,-oo,oo))
print("∫∫_ℝ² exp(-((x-1)²+(y+2)²)) =", s2d)   # π


∫∫_ℝ² exp(-(x²+y²))         = pi
∫∫_ℝ² exp(-(2x²+3y²))       = sqrt(6)*pi/6
∫∫∫_ℝ³ exp(-(x²+y²+z²))     = pi**(3/2)
∫∫_ℝ² exp(-((x-1)²+(y+2)²)) = pi


### 15.3  Strategy 3 — Quadratic, even, half-infinite

Same quadratic engine as S2, but some dimensions have range $[0,\infty)$.
Requires $f\circ g$ to be even in those dimensions; result is halved for each.


In [27]:
# Half-line Gaussian: ∫_0^∞ exp(-x²) dx = √π/2
s3a = multiple_integrate(exp(-x**2), (x, 0, oo))
print("∫₀^∞ exp(-x²)          =", s3a)   # √π/2

# Quarter-plane: ∫∫_[0,∞)² exp(-(x²+y²)) = π/4
s3b = multiple_integrate(exp(-(x**2+y**2)), (x, 0, oo), (y, 0, oo))
print("∫∫_[0,∞)² exp(-(x²+y²)) =", s3b)   # π/4

# Mixed full/half: π/2
s3c = multiple_integrate(exp(-(x**2+y**2)), (x, -oo, oo), (y, 0, oo))
print("∫_ℝ∫₀^∞ exp(-(x²+y²))  =", s3c)   # π/2


∫₀^∞ exp(-x²)          = sqrt(pi)/2
∫∫_[0,∞)² exp(-(x²+y²)) = pi/4
∫_ℝ∫₀^∞ exp(-(x²+y²))  = pi/2


### 15.4  Strategy 4 — General polynomial, Heaviside layer-cake

For polynomial $g$ on any bounded or semi-infinite domain.
SymPy integrates $\Theta(y-g(\mathbf{x}))$ dimension by dimension to obtain $\mu(y)$.


In [28]:
# Bounded domain
s4a = multiple_integrate(x**2 * y, (x, 0, 1), (y, 0, 2))
print("∫∫ x²y  [0,1]×[0,2]      =", s4a)   # 2/3

# (x+y)³ on unit square
s4b = multiple_integrate((x+y)**3, (x, 0, 1), (y, 0, 1))
print("∫∫ (x+y)³  [0,1]²        =", s4b)

# Triangle domain with variable upper limit
s4c = multiple_integrate(x**2 * y, (y, 0, 1-x), (x, 0, 1))
print("∫∫ x²y  triangle         =", s4c)   # 1/60

# Triple integral
s4d = multiple_integrate(x*y*z, (x,0,1),(y,0,1),(z,0,1))
print("∫∫∫ xyz  [0,1]³          =", s4d)   # 1/8


∫∫ x²y  [0,1]×[0,2]      = 2/3
∫∫ (x+y)³  [0,1]²        = 3/2
∫∫ x²y  triangle         = 1/60
∫∫∫ xyz  [0,1]³          = 1/8


### 15.5  Strategy 5 — Separable non-polynomial $g$

**When $g(x_1,\ldots,x_n) = h_1(x_1)+\cdots+h_n(x_n)$** (each term depends on one variable),
the pushforward density of the sum is the **convolution** of the marginal densities:

$$\mu'(y) = (\nu_1 * \nu_2 * \cdots * \nu_n)(y)$$

Each $\nu_i$ is computed via a 1-D Heaviside integral. This works for **any** single-variable
functions $h_i$ — trig, exponential, logarithm, etc.


In [29]:
# Trig sum: ∫∫_[0,π]² cos(x+y) dx dy = -4
s5a = multiple_integrate(cos(x + y), (x, 0, pi), (y, 0, pi))
print("∫∫ cos(x+y)  [0,π]²      =", s5a)   # -4

# Exponential sum: ∫∫_[0,∞)² exp(-(x+y)) = 1
s5b = multiple_integrate(exp(-(x + y)), (x, 0, oo), (y, 0, oo))
print("∫∫ exp(-(x+y))  [0,∞)²  =", s5b)   # 1

# Squared trig sum
s5c = multiple_integrate((sin(x) + sin(y))**2, (x, 0, 1), (y, 0, 1))
print("∫∫ (sin x + sin y)²     =", simplify(s5c))

# 3-D trig sum
s5d = multiple_integrate(sin(x + y + z), (x, 0, 1), (y, 0, 1), (z, 0, 1))
print("∫∫∫ sin(x+y+z)  [0,1]³  =", simplify(s5d))


∫∫ cos(x+y)  [0,π]²      = -4
∫∫ exp(-(x+y))  [0,∞)²  = 1
∫∫ (sin x + sin y)²     = -4*cos(1) - sin(2)/2 + cos(2) + 4
∫∫∫ sin(x+y+z)  [0,1]³  = -1 + cos(3) - 3*cos(2) + 3*cos(1)


### 15.6  Strategy 6 — Monotone substitution

For **single-variable non-polynomial $g$** with no interior critical points,
the co-area density is $\mu'(y) = |dx/dy| = 1/|g'(g^{-1}(y))|$.

The library inverts $g$ analytically via `sympy.solve` and integrates $f(y)|dx/dy|$.
This handles exponentials, logarithms, rational functions, algebraic functions, and more.


In [30]:
# Exponential: ∫_0^1 exp(x) dx = e - 1
s6a = multiple_integrate(exp(x), (x, 0, 1))
print("∫₀¹ exp(x)       =", s6a)   # E - 1

# Logarithm: ∫_1^e log(x) dx = 1
s6b = multiple_integrate(log(x), (x, 1, E))
print("∫₁ᵉ log(x)       =", s6b)   # 1

# Integrable log singularity: ∫_0^1 log(x) dx = -1
s6c = multiple_integrate(log(x), (x, 0, 1))
print("∫₀¹ log(x)       =", s6c)   # -1

# Arctan: ∫_0^1 1/(1+x²) dx = π/4
s6d = multiple_integrate(1/(1 + x**2), (x, 0, 1))
print("∫₀¹ 1/(1+x²)     =", s6d)   # π/4

# Algebraic: ∫_0^4 √x dx = 16/3
s6e = multiple_integrate(sqrt(x), (x, 0, 4))
print("∫₀⁴ √x           =", s6e)   # 16/3

# Half-line (monotone decreasing): ∫_0^∞ exp(-x) dx = 1
s6f = multiple_integrate(exp(-x), (x, 0, oo))
print("∫₀^∞ exp(-x)     =", s6f)   # 1

# With free dimension: ∫₀¹∫₀¹ exp(-x) dx dy = 1 - 1/e
s6g = multiple_integrate(exp(-x), (x, 0, 1), (y, 0, 1))
print("∫₀¹∫₀¹ exp(-x)   =", s6g)   # 1 - 1/e


∫₀¹ exp(x)       = -1 + E
∫₁ᵉ log(x)       = 1
∫₀¹ log(x)       = -1
∫₀¹ 1/(1+x²)     = pi/4
∫₀⁴ √x           = 16/3
∫₀^∞ exp(-x)     = 1
∫₀¹∫₀¹ exp(-x)   = 1 - exp(-1)


### 15.7  Strategy 7 — Piecewise-monotone substitution

When $g$ has interior critical points, the domain is split there.
Strategy 6 is applied on each monotone piece and the results are summed.

The library also detects **non-differentiable kinks** (zeros of `Abs` arguments
and `sqrt` radicands), so `|x|`, `|sin(x)|`, etc. are handled correctly.


In [31]:
# sin(x) on [0,π]: critical point at π/2
s7a = multiple_integrate(sin(x), (x, 0, pi))
print("∫₀^π sin(x)          =", s7a)   # 2

# cos(x) on [0,2π]: critical point at π
s7b = multiple_integrate(cos(x), (x, 0, 2*pi))
print("∫₀^2π cos(x)         =", s7b)   # 0

# sin²(x) on [0,π]
s7c = multiple_integrate(sin(x)**2, (x, 0, pi))
print("∫₀^π sin²(x)         =", s7c)   # π/2

# Non-analytic f: |x| on [-1,1]  (kink at 0)
s7d = multiple_integrate(Abs(x), (x, -1, 1))
print("∫₋₁¹ |x|             =", s7d)   # 1

# |sin(x)| on [0,2π]
s7e = multiple_integrate(Abs(sin(x)), (x, 0, 2*pi))
print("∫₀^2π |sin(x)|       =", s7e)   # 4

# |cos(x)| on [0,π]
s7f = multiple_integrate(Abs(cos(x)), (x, 0, pi))
print("∫₀^π |cos(x)|        =", s7f)   # 2

# With free dimension: ∫₀^π∫₀¹ sin(x) dy dx = 2
s7g = multiple_integrate(sin(x), (x, 0, pi), (y, 0, 1))
print("∫₀^π∫₀¹ sin(x) dy dx =", s7g)   # 2


∫₀^π sin(x)          = 2
∫₀^2π cos(x)         = 0
∫₀^π sin²(x)         = pi/2
∫₋₁¹ |x|             = 1
∫₀^2π |sin(x)|       = 4
∫₀^π |cos(x)|        = 2
∫₀^π∫₀¹ sin(x) dy dx = 2


### 15.8  Strategy 8 — General non-polynomial, Heaviside layer-cake

For **multi-variable non-polynomial $g$** that is not additively separable (S5).
SymPy evaluates $\int_\Omega\Theta(y-g(\mathbf{x}))\,d\mathbf{x}$ directly.


In [32]:
# Product of single-variable functions (not a sum → not S5)
s8a = multiple_integrate(sin(x)*cos(y), (x, 0, pi/2), (y, 0, pi/2))
print("∫∫ sin(x)cos(y)  [0,π/2]²  =", s8a)   # 1

# Heaviside of a linear combination: area where x+y > 2 in [0,2]²
s8b = multiple_integrate(Heaviside(x + y - 2), (x, 0, 2), (y, 0, 2))
print("∫∫ Θ(x+y-2)  [0,2]²        =", s8b)   # 2


∫∫ sin(x)cos(y)  [0,π/2]²  = 1
∫∫ Θ(x+y-2)  [0,2]²        = 2


### 15.9  Strategy 9 — Fallback (plain iterated integration)

When no earlier strategy applies, `multiple_integrate` falls through to plain
iterated `sympy.integrate`. This handles products like $x\sin(x)$, expressions with
variable limits, and integrands that don't factorise as $f(g(\mathbf{x}))$.


In [33]:
# x·sin(x): not of f(g) form — integration by parts needed
s9a = multiple_integrate(x * sin(x), (x, 0, pi))
print("∫₀^π x·sin(x)            =", s9a)   # π

# x·log(x)
s9b = multiple_integrate(x * log(x), (x, 0, 1))
print("∫₀¹ x·log(x)             =", s9b)   # -1/4

# Variable limits (triangle)
s9c = multiple_integrate(x + y, (y, 0, 1 - x), (x, 0, 1))
print("∫∫_triangle (x+y)        =", s9c)   # 1/3

# sin(x)·cos(y) on [0,π]² — separable product that integrates to 0
s9d = multiple_integrate(sin(x)*cos(y), (x, 0, pi), (y, 0, pi))
print("∫∫ sin(x)·cos(y)  [0,π]² =", s9d)   # 0


∫₀^π x·sin(x)            = pi
∫₀¹ x·log(x)             = -1/4
∫∫_triangle (x+y)        = 1/3
∫∫ sin(x)·cos(y)  [0,π]² = 0


### 15.10  Mixed and cross-cutting examples

These combine multiple hard properties: non-analytic $f$, non-polynomial $g$,
2-D non-rectangular domains, and improper integrals.


In [34]:
# Non-analytic f + non-polynomial g: |x - y| on [0,1]²
m1 = multiple_integrate(Abs(x - y), (x, 0, 1), (y, 0, 1))
print("∫∫ |x-y|  [0,1]²              =", m1)   # 1/3

# Discontinuous f: Heaviside(sin(x) - 1/2) on [0,π]
# sin(x) > 1/2 on (π/6, 5π/6) — length 2π/3
m2 = multiple_integrate(Heaviside(sin(x) - Rational(1, 2)), (x, 0, pi))
print("∫₀^π Θ(sin x - 1/2)          =", m2)   # 2π/3

# exp(-|x|) on [-1,1] = 2(1 - e^{-1})
m3 = multiple_integrate(exp(-Abs(x)), (x, -1, 1))
print("∫₋₁¹ exp(-|x|)               =", simplify(m3))

# 3-D mixed: ∫₀^π∫₀¹∫₀^∞ sin(x)·exp(-y)·exp(-z)
m4 = multiple_integrate(sin(x)*exp(-y)*exp(-z), (x,0,pi),(y,0,1),(z,0,oo))
print("∫₀^π∫₀¹∫₀^∞ sin·e⁻ʸ·e⁻ᶻ    =", simplify(m4))   # 2(1 - 1/e)

# Physics: moment of inertia of unit cube about z-axis
m5 = multiple_integrate(x**2 + y**2, (x,0,1),(y,0,1),(z,0,1))
print("I_z of unit cube              =", m5)   # 2/3

# Probability: E[(X+Y)²] for X,Y ~ Uniform[0,1]
m6 = multiple_integrate((x+y)**2, (x,0,1),(y,0,1))
print("E[(X+Y)²]                     =", m6)   # 7/6


∫∫ |x-y|  [0,1]²              = 1/3
∫₀^π Θ(sin x - 1/2)          = 2*pi/3
∫₋₁¹ exp(-|x|)               = 2 - 2*exp(-1)
∫₀^π∫₀¹∫₀^∞ sin·e⁻ʸ·e⁻ᶻ    = 2 - 2*exp(-1)
I_z of unit cube              = 2/3
E[(X+Y)²]                     = 7/6


### 15.11  Convergent and divergent improper integrals

`multiple_integrate` does not pre-screen for convergence — divergent integrals
propagate as `oo` or remain unevaluated.


In [35]:
# Convergent: p-test ∫_1^∞ x^{-2} dx = 1  (p = -2 < -1)
c1 = multiple_integrate(x**(-2), (x, 1, oo))
print("∫₁^∞ x⁻² dx =", c1)   # 1

# Convergent: integrable singularity ∫_0^1 x^{-1/2} dx = 2
c2 = multiple_integrate(x**Rational(-1, 2), (x, 0, 1))
print("∫₀¹ x^(-1/2) dx =", c2)   # 2

# Convergent: log singularity ∫_0^1 log(x) dx = -1
c3 = multiple_integrate(log(x), (x, 0, 1))
print("∫₀¹ log(x) dx =", c3)   # -1

# Divergent: p-test boundary ∫_1^∞ 1/x dx = ∞
d1 = multiple_integrate(1/x, (x, 1, oo))
print("∫₁^∞ 1/x dx =", d1)   # oo

# Divergent: non-integrable singularity ∫_0^1 1/x dx = ∞
d2 = multiple_integrate(1/x, (x, 0, 1))
print("∫₀¹ 1/x dx =", d2)   # oo

# Divergent: wrong-sign Gaussian
d3 = multiple_integrate(exp(x**2), (x, -oo, oo))
print("∫_ℝ exp(x²) dx =", d3)   # oo


∫₁^∞ x⁻² dx = 1
∫₀¹ x^(-1/2) dx = 2
∫₀¹ log(x) dx = -1
∫₁^∞ 1/x dx = oo
∫₀¹ 1/x dx = oo
∫_ℝ exp(x²) dx = oo


### 15.12  Strategy routing — timing comparison

In [36]:
import time

cases = [
    ("S1  Linear [0,∞)²",      exp(-(x+y)),                   (x,0,oo),(y,0,oo)),
    ("S2  Gaussian ℝ²",         exp(-(x**2+y**2)),              (x,-oo,oo),(y,-oo,oo)),
    ("S3  Half-Gaussian [0,∞)", exp(-x**2),                    (x,0,oo)),
    ("S4  Poly unit square",    (x+y)**3,                      (x,0,1),(y,0,1)),
    ("S5  cos(x+y) [0,π]²",    cos(x+y),                      (x,0,pi),(y,0,pi)),
    ("S6  log(x) [1,e]",        log(x),                        (x,1,E)),
    ("S6  1/(1+x²) [0,1]",      1/(1+x**2),                    (x,0,1)),
    ("S7  sin(x) [0,π]",        sin(x),                        (x,0,pi)),
    ("S7  |x| [-1,1]",          Abs(x),                        (x,-1,1)),
    ("S9  x·sin(x) [0,π]",      x*sin(x),                      (x,0,pi)),
]

for label, integrand, *ranges in cases:
    t0 = time.perf_counter()
    result = multiple_integrate(integrand, *ranges)
    ms = (time.perf_counter() - t0) * 1000
    print(f"{label:30s}  →  {str(result):15s}  ({ms:.1f} ms)")


S1  Linear [0,∞)²               →  1                (3.1 ms)
S2  Gaussian ℝ²                 →  pi               (2.9 ms)
S3  Half-Gaussian [0,∞)         →  sqrt(pi)/2       (5.6 ms)
S4  Poly unit square            →  3/2              (5.1 ms)
S5  cos(x+y) [0,π]²             →  -4               (14.3 ms)
S6  log(x) [1,e]                →  1                (0.2 ms)
S6  1/(1+x²) [0,1]              →  pi/4             (2.5 ms)
S7  sin(x) [0,π]                →  2                (2.9 ms)
S7  |x| [-1,1]                  →  1                (0.2 ms)
S9  x·sin(x) [0,π]              →  pi               (8.2 ms)


## <a id="sec-16-supported-families"></a>16  Supported Families — One Example Each

This section gives a **compact tour of the currently supported families** that the solver is designed to handle well.  
These are **not** regression tests; the point is to show one representative exact integral for each family and the kind
of closed form the library can return.

The examples below cover:

1. constants and zero integrands  
2. basic one-dimensional exact integrals  
3. endpoint-singular but convergent integrals  
4. rational full-line integrals  
5. box moments  
6. Gaussian moments  
7. radial / polar-coordinate examples  
8. trigonometric and exponential transform-friendly cases  
9. special-function outputs such as logarithms, arctangents, beta, and gamma values

### 16.1  Constants and zero integrands

A symbolic multiple-integration engine should handle constants and zero integrands immediately. On rectangular regions,
these reduce to geometric volume factors.

In [37]:
const_2d = multiple_integrate(1, (x, 0, 1), (y, 0, 2))
zero_2d  = multiple_integrate(0, (x, -1, 3), (y, 2, 5))

print("∫∫ 1  [0,1]×[0,2] =", const_2d)   # 2
print("∫∫ 0  [-1,3]×[2,5] =", zero_2d)   # 0

∫∫ 1  [0,1]×[0,2] = 2
∫∫ 0  [-1,3]×[2,5] = 0


### 16.2  Basic one-dimensional exact integrals

These are the simplest exact cases, but they are still important because they exercise the same dispatch logic used by
higher-dimensional problems.

In [38]:
basic_poly = multiple_integrate(x**3, (x, 0, 2))
basic_trig = multiple_integrate(sin(x), (x, 0, pi))
basic_exp  = multiple_integrate(exp(x), (x, 0, 1))

print("∫₀² x³ dx     =", basic_poly)   # 4
print("∫₀^π sin(x) dx =", basic_trig)  # 2
print("∫₀¹ eˣ dx     =", basic_exp)    # E - 1

∫₀² x³ dx     = 4
∫₀^π sin(x) dx = 2
∫₀¹ eˣ dx     = -1 + E


### 16.3  Endpoint-singular but convergent integrals

Not every singular integrand diverges. These examples are finite even though the integrand is unbounded at an endpoint.

In [39]:
singular_a = multiple_integrate(x**Rational(-1, 2), (x, 0, 1))
singular_b = multiple_integrate(log(x), (x, 0, 1))
singular_c = multiple_integrate(1 / sqrt(1 - x), (x, 0, 1))

print("∫₀¹ x^(-1/2) dx      =", singular_a)  # 2
print("∫₀¹ log(x) dx        =", singular_b)  # -1
print("∫₀¹ 1/sqrt(1-x) dx   =", singular_c)  # 2

∫₀¹ x^(-1/2) dx      = 2
∫₀¹ log(x) dx        = -1
∫₀¹ 1/sqrt(1-x) dx   = 2


### 16.4  Rational full-line integrals

These are classic exact integrals whose answers involve \(\pi\). They are good representatives for rational kernels on
the whole real line.

In [40]:
rat_a = multiple_integrate(1 / (x**2 + 1), (x, -oo, oo))
rat_b = multiple_integrate(1 / (x**4 + 1), (x, -oo, oo))

print("∫_ℝ dx/(x²+1)   =", rat_a)  # π
print("∫_ℝ dx/(x⁴+1)   =", rat_b)  # π/sqrt(2)

∫_ℝ dx/(x²+1)   = pi
∫_ℝ dx/(x⁴+1)   = sqrt(2)*pi/2


### 16.5  Box moments

On product regions such as rectangles and boxes, polynomial moments are among the most natural exact multiple integrals.

In [41]:
box_a = multiple_integrate(x**2 * y**3, (x, 0, 1), (y, 0, 1))
box_b = multiple_integrate(x**2 * y**2, (x, -1, 1), (y, -1, 1))

print("∫∫ x²y³  [0,1]²     =", box_a)  # 1/12
print("∫∫ x²y²  [-1,1]²    =", box_b)  # 4/9

∫∫ x²y³  [0,1]²     = 1/12
∫∫ x²y²  [-1,1]²    = 4/9


### 16.6  Gaussian moments

Gaussian integrals are one of the central exact families in analysis, probability, and physics. The multidimensional
examples below also illustrate separability and even/odd moment structure.

In [42]:
gauss_1d = multiple_integrate(exp(-x**2), (x, -oo, oo))
gauss_1d_moment = multiple_integrate(x**2 * exp(-x**2), (x, -oo, oo))
gauss_2d = multiple_integrate(exp(-(x**2 + y**2)), (x, -oo, oo), (y, -oo, oo))

print("∫_ℝ e^(-x²) dx              =", gauss_1d)         # sqrt(pi)
print("∫_ℝ x² e^(-x²) dx           =", gauss_1d_moment)  # sqrt(pi)/2
print("∫∫_ℝ² e^(-(x²+y²)) dx dy    =", gauss_2d)         # pi

∫_ℝ e^(-x²) dx              = sqrt(pi)
∫_ℝ x² e^(-x²) dx           = sqrt(pi)/2
∫∫_ℝ² e^(-(x²+y²)) dx dy    = pi


### 16.7  Radial / polar-coordinate examples

The current solver does **not** implement a completely general automatic polar-coordinate engine, but radial families are
still very natural examples to include in the notebook. Writing them directly in polar coordinates yields compact exact
results.

In [43]:
disk_area = integrate(1 * r, (r, 0, 1), (theta, 0, 2*pi))
disk_x2 = integrate((r*cos(theta))**2 * r, (r, 0, 1), (theta, 0, 2*pi))
disk_radial = integrate(r**2 * r, (r, 0, 1), (theta, 0, 2*pi))

print("∬_{x²+y²≤1} 1 dA           =", simplify(disk_area))    # pi
print("∬_{x²+y²≤1} x² dA          =", simplify(disk_x2))      # pi/4
print("∬_{x²+y²≤1} (x²+y²) dA     =", simplify(disk_radial))  # pi/2

∬_{x²+y²≤1} 1 dA           = pi
∬_{x²+y²≤1} x² dA          = pi/4
∬_{x²+y²≤1} (x²+y²) dA     = pi/2


### 16.8  Trigonometric and exponential transform-friendly cases

Sums such as \(x+y\) or damped trigonometric/exponential expressions often simplify after convolution-style reasoning,
trigonometric identities, or straightforward separability.

In [44]:
trig_sep = multiple_integrate(sin(x) * sin(y), (x, 0, pi), (y, 0, pi))
trig_sum = multiple_integrate(cos(x + y), (x, 0, pi), (y, 0, pi))
exp_sum = multiple_integrate(exp(-(x + y)), (x, 0, oo), (y, 0, oo))
damped_cos = multiple_integrate(exp(-x) * cos(x), (x, 0, oo))

print("∫∫ sin(x)sin(y)  [0,π]²    =", trig_sep)   # 4
print("∫∫ cos(x+y)  [0,π]²        =", trig_sum)   # -4
print("∫∫ e^(-(x+y))  [0,∞)²      =", exp_sum)    # 1
print("∫₀^∞ e^(-x) cos(x) dx      =", damped_cos) # 1/2

∫∫ sin(x)sin(y)  [0,π]²    = 4
∫∫ cos(x+y)  [0,π]²        = -4
∫∫ e^(-(x+y))  [0,∞)²      = 1
∫₀^∞ e^(-x) cos(x) dx      = 1/2


### 16.9  Special-function outputs

A useful symbolic integrator should return exact answers involving logarithms, inverse trigonometric functions, beta
functions, and gamma functions when those are the natural closed forms.

In [45]:
spec_log = multiple_integrate(1 / (1 + x), (x, 0, 1))
spec_atan = multiple_integrate(1 / (1 + x**2), (x, 0, 1))
beta_ex = multiple_integrate(x**2 * (1 - x)**3, (x, 0, 1))
gamma_ex = multiple_integrate(sqrt(x) * exp(-x), (x, 0, oo))

print("∫₀¹ dx/(1+x)               =", spec_log)  # log(2)
print("∫₀¹ dx/(1+x²)              =", spec_atan) # pi/4
print("∫₀¹ x²(1-x)³ dx            =", beta_ex)   # 1/60
print("∫₀^∞ sqrt(x)e^(-x) dx      =", gamma_ex)  # sqrt(pi)/2

∫₀¹ dx/(1+x)               = log(2)
∫₀¹ dx/(1+x²)              = pi/4
∫₀¹ x²(1-x)³ dx            = 1/60
∫₀^∞ sqrt(x)e^(-x) dx      = sqrt(pi)/2


### 16.10  A note on dependent bounds

The project can successfully evaluate some iterated integrals whose inner bounds depend on outer variables when the integral is easy in the order given. It also has stronger support than before for standard simplices, disks, balls, shells, and selected affine images of those regions when a recognized exact formula or coordinate change applies.

Even so, the current implementation should **not** be viewed as a full general-purpose engine for arbitrary geometric region rewriting or unrestricted order reversal. For that reason, this notebook keeps the supported-family overview focused on families that are currently well supported and stable in ordinary use.


## <a id="sec-17-complicated-examples"></a>17  Complicated-looking exact multiple integrals

This section collects a batch of **more elaborate-looking but still highly structured** examples that `multiple_integrate` can now handle much more directly than before. They are useful as stress tests because many of them look intimidating at first glance, yet reduce to recognizable families such as:

- simplex / Dirichlet integrals,
- product Gamma and Beta integrals,
- Gaussian moments and quadratic-form Gaussians,
- polar or spherical coordinate-change examples,
- damped oscillatory integrals on half-infinite regions.

The next code cell evaluates a selection of these examples and prints the exact symbolic results.


In [46]:
w = sp.symbols('w', positive=True)

def _showcase(name, thunk):
    print(f"{name}:")
    try:
        value = thunk()
        sp.pprint(sp.simplify(value))
    except Exception as error:
        print(f"  ERROR: {type(error).__name__}: {error}")
    print()

complicated_cases = [
    (
        "Simplex Dirichlet (half-integer exponents)",
        lambda: multiple_integrate(
            sp.sqrt(x) * y**sp.Rational(3, 2) * sp.sqrt(z) * sp.sqrt(1 - x - y - z),
            (z, 0, 1 - x - y),
            (y, 0, 1 - x),
            (x, 0, 1),
        ),
    ),
    (
        "Simplex polynomial weight",
        lambda: multiple_integrate(
            x * y * z * (1 - x - y - z)**2,
            (z, 0, 1 - x - y),
            (y, 0, 1 - x),
            (x, 0, 1),
        ),
    ),
    (
        "Simplex inverse square-root singularity",
        lambda: multiple_integrate(
            1 / sp.sqrt(x * y * z * (1 - x - y - z)),
            (z, 0, 1 - x - y),
            (y, 0, 1 - x),
            (x, 0, 1),
        ),
    ),
    (
        "Positive-quadrant Gamma product",
        lambda: multiple_integrate(
            x**sp.Rational(1, 2) * y**sp.Rational(3, 2) * sp.exp(-(x + y)),
            (y, 0, sp.oo),
            (x, 0, sp.oo),
        ),
    ),
    (
        "Triple Gamma product with scaled exponentials",
        lambda: multiple_integrate(
            x**2 * y * z**3 * sp.exp(-x - 2*y - 3*z),
            (z, 0, sp.oo),
            (y, 0, sp.oo),
            (x, 0, sp.oo),
        ),
    ),
    (
        "Cube product Beta integral",
        lambda: multiple_integrate(
            x**sp.Rational(1, 2) / sp.sqrt(1 - x)
            * y**sp.Rational(1, 2) / sp.sqrt(1 - y)
            * z**sp.Rational(1, 2) / sp.sqrt(1 - z),
            (z, 0, 1),
            (y, 0, 1),
            (x, 0, 1),
        ),
    ),
    (
        "Mixed Beta product on the unit square",
        lambda: multiple_integrate(
            x**sp.Rational(1, 3) * (1 - x)**sp.Rational(1, 2) * y**2 * (1 - y)**3,
            (y, 0, 1),
            (x, 0, 1),
        ),
    ),
    (
        "Rational Dirichlet integral on (0, ∞)^2",
        lambda: multiple_integrate(
            (x * y)**sp.Rational(1, 3) / (1 + x + y)**4,
            (y, 0, sp.oo),
            (x, 0, sp.oo),
        ),
    ),
    (
        "Gaussian moment, factored anisotropic kernel",
        lambda: multiple_integrate(
            x**4 * y**2 * sp.exp(-2*x**2 - 3*y**2),
            (y, -sp.oo, sp.oo),
            (x, -sp.oo, sp.oo),
        ),
    ),
    (
        "Correlated Gaussian quadratic form",
        lambda: multiple_integrate(
            sp.exp(-(x**2 + 2*x*y + 5*y**2)),
            (y, -sp.oo, sp.oo),
            (x, -sp.oo, sp.oo),
        ),
    ),
    (
        "Radial Gaussian moment in 3D",
        lambda: multiple_integrate(
            (x**2 + y**2 + z**2) * sp.exp(-(x**2 + y**2 + z**2)),
            (z, -sp.oo, sp.oo),
            (y, -sp.oo, sp.oo),
            (x, -sp.oo, sp.oo),
        ),
    ),
    (
        "Difference-variable Gaussian moment",
        lambda: multiple_integrate(
            (x - y)**4 * sp.exp(-(x**2 + y**2)),
            (y, -sp.oo, sp.oo),
            (x, -sp.oo, sp.oo),
        ),
    ),
    (
        "Rotated Gaussian with x^2 factor",
        lambda: multiple_integrate(
            x**2 * sp.exp(-(x + y)**2 - (x - y)**2),
            (y, -sp.oo, sp.oo),
            (x, -sp.oo, sp.oo),
        ),
    ),
    (
        "Damped cosine over the positive quadrant",
        lambda: multiple_integrate(
            sp.exp(-x - y) * sp.cos(x - y),
            (y, 0, sp.oo),
            (x, 0, sp.oo),
        ),
    ),
    (
        "Triple damped cosine over the positive orthant (may fail in current SymPy fallback)",
        lambda: multiple_integrate(
            sp.exp(-(x + y + z)) * sp.cos(x + y + z),
            (z, 0, sp.oo),
            (y, 0, sp.oo),
            (x, 0, sp.oo),
        ),
    ),
    (
        "Oscillatory product moment",
        lambda: multiple_integrate(
            x * y * sp.exp(-x - y) * sp.sin(x) * sp.sin(y),
            (y, 0, sp.oo),
            (x, 0, sp.oo),
        ),
    ),
    (
        "Four-simplex Dirichlet integral",
        lambda: multiple_integrate(
            x**sp.Rational(1, 2)
            * y**sp.Rational(1, 3)
            * z**sp.Rational(2, 3)
            * w**sp.Rational(1, 4)
            * (1 - x - y - z - w)**sp.Rational(3, 2),
            (w, 0, 1 - x - y - z),
            (z, 0, 1 - x - y),
            (y, 0, 1 - x),
            (x, 0, 1),
        ),
    ),
]

for name, thunk in complicated_cases:
    _showcase(name, thunk)

Simplex Dirichlet (half-integer exponents):
  2 
 π  
────
7680

Simplex polynomial weight:
1/20160

Simplex inverse square-root singularity:
1                                                   
⌠                                                   
⎮ 1 - x -x - y + 1                                  
⎮   ⌠       ⌠                                       
⎮   ⎮       ⎮                 1                     
⎮   ⎮       ⎮      ──────────────────────── dz dy   
⎮   ⎮       ⎮        ______________________         
⎮   ⎮       ⎮      ╲╱ y⋅z⋅(-x - y - z + 1)          
⎮   ⌡       ⌡                                       
⎮   0       0                                       
⎮ ─────────────────────────────────────────────── dx
⎮                       √x                          
⌡                                                   
0                                                   

Positive-quadrant Gamma product:
3⋅π
───
 8 

Triple Gamma product with scaled exponentials:
1/27

Cube product Bet

## <a id="sec-18-convergence-applications-limitations"></a>18  Convergence, applications, and limitations

### 18.1  Endpoint singularities and integrability

A singular integrand need not be divergent. For example,
$$
\int_0^1 x^{-1/2}\,dx
$$
converges, whereas
$$
\int_0^1 x^{-1}\,dx
$$
does not. In several variables, geometry matters as much as the local blow-up rate: a factor that looks singular in Cartesian coordinates may become harmless after the correct Jacobian is included.

This is one reason symbolic multiple integration benefits from recognizing the region and not just the formula for the integrand.

### 18.2  Where these exact families arise

The examples in this notebook are not merely artificial exercises.

- **Gaussian integrals** appear in probability, statistics, heat kernels, and quantum mechanics.
- **Simplex / Dirichlet integrals** appear in Bayesian modeling and normalization of Dirichlet distributions.
- **Disk and ball integrals** appear in mechanics, electrostatics, and PDEs with radial symmetry.
- **Oscillatory–exponential integrals** appear in Laplace and Fourier analysis, signal processing, and Green-function calculations.
- **Layer-cake and co-area viewpoints** appear in geometric measure theory, rearrangement inequalities, and distribution-function methods.

### 18.3  When symbolic multiple integration becomes hard

Even strong structure-aware methods have limits. Difficulty rises quickly for:

- highly non-algebraic or branch-sensitive region boundaries;
- conditionally convergent integrals where order changes are delicate;
- piecewise geometry with many interacting cases;
- high-dimensional integrals that do not match a recognizable canonical family.

For that reason, the package is best viewed as a **recognition-driven exact integrator for structured families**, not as a universal solver for every iterated integral that can be written down.

## <a id="sec-19-references"></a>19  References

### Textbooks

1. **Apostol, T. M.** (1969). *Calculus, Vol. 2: Multi-Variable Calculus and Linear Algebra*. Wiley.  
   Rigorous treatment of multiple integrals and Fubini's theorem.

2. **Rudin, W.** (1987). *Real and Complex Analysis* (3rd ed.). McGraw-Hill.  
   Chapter 8: Integration on product spaces, Fubini–Tonelli theorem.

3. **Evans, L. C., & Gariepy, R. F.** (2015). *Measure Theory and Fine Properties of Functions* (revised ed.). CRC Press.  
   Co-area formula: Theorem 3.11.

4. **Folland, G. B.** (1999). *Real Analysis: Modern Techniques and Their Applications* (2nd ed.). Wiley.  
   Layer-cake representation: Proposition 6.16.

5. **Federer, H.** (1969). *Geometric Measure Theory*. Springer.  
   Original formulation of the co-area formula.

### Papers & technical references

6. **Bronstein, M.** (2005). *Symbolic Integration I: Transcendental Functions* (2nd ed.). Springer.  
   Algorithms underlying SymPy's univariate `integrate`.

7. **Moses, J.** (1971). Symbolic integration: The stormy decade. *Communications of the ACM*, 14(8), 548–560.  
   Historical context for computer algebra integration algorithms.

8. **Risch, R. H.** (1969). The problem of integration in finite terms. *Transactions of the American Mathematical Society*, 139, 167–189.  
   The Risch algorithm used for indefinite integration in SymPy.

### Software

9. **SymPy Development Team** (2023). *SymPy: Python library for symbolic mathematics*. https://www.sympy.org  

### Online resources

10. Evans, L. (2010). *Measure theory and integration* (lecture notes). UC Berkeley.  
    Clear exposition of the layer-cake formula with proofs.

11. Strichartz, R. S. (1994). *A Guide to Distribution Theory and Fourier Transforms*. CRC Press.  
    Heaviside function, delta distributions, and their use in integration.